# 19 — Submission: P7c + duración de la publicación

`18_barrido_campos.ipynb` encontró que `duracion_dias` (`stop_time` − `start_time`) suma de verdad: las publicaciones que duran **60 días + unos segundos** son casi todas `used` (artefacto de cómo la plataforma calcula el vencimiento de las publicaciones creadas por primera vez). Sumada a los dos modelos de P7c:
- semilla de CV 42: **0,9420** (P7c 0,9376), 5/5 folds, +0,0044;
- semilla de CV 7: **0,9422** (P7c 0,9373), 5/5 folds, +0,0049.

Esta notebook entrena los dos modelos (CatBoost P6b y HistGB P7a de `features.py`, cada uno con `duracion_dias` como numérica extra) con todo el train, promedia 50/50 y genera la submission. **No toca `features.py`** (la duración se agrega acá; pasarla a `features.py` queda pendiente). **No sube nada.**

In [1]:
import sys
sys.path.append("..")
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

from features import (aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
                      armar_pipeline_final, armar_pipeline_histgb_combos, COLUMNAS_NUMERICAS, INDICES_CAT)


def preparar(df):
    df = agregar_combos(agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(df))))
    df["duracion_dias"] = (pd.to_datetime(df["stop_time"]) - pd.to_datetime(df["start_time"])).dt.total_seconds() / 86400
    return df


train = pd.read_json("../data/train_data.jsonlines", lines=True)
test = pd.read_json("../data/test_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train, test = preparar(train), preparar(test)
print("60 días exactos -> train:", round((train["duracion_dias"] == 60).mean(), 4), "| test:", round((test["duracion_dias"] == 60).mean(), 4))

60 días exactos -> train: 0.9447 | test: 0.9421


## Los dos modelos con la duración

- **CatBoost P6b**: `armar_pipeline_final()` con `duracion_dias` sumada a las numéricas. Las categóricas se corren un lugar (`INDICES_CAT + 1`).
- **HistGB P7a**: `armar_pipeline_histgb_combos()` con `duracion_dias` como columna extra al final.

In [2]:
catboost = armar_pipeline_final()
catboost.named_steps["preprocesador"].transformers[0] = ("num", "passthrough", COLUMNAS_NUMERICAS + ["duracion_dias"])
indices_cat = [i + 1 for i in INDICES_CAT]

histgb = armar_pipeline_histgb_combos()
histgb.named_steps["preprocesador"].transformers.append(("duracion", "passthrough", ["duracion_dias"]))

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    catboost.fit(train, y, modelo__cat_features=indices_cat)
    histgb.fit(train, y)
print("Categóricas de CatBoost:", catboost.named_steps["modelo"].get_cat_feature_indices())

Categóricas de CatBoost: [526, 527, 528, 529, 530, 531]


## Promedio 50/50 y CSV

In [3]:
prob_new = 0.5 * catboost.predict_proba(test)[:, 1] + 0.5 * histgb.predict_proba(test)[:, 1]
submission = pd.DataFrame({"ID": np.arange(1, len(test) + 1), "condition": np.where(prob_new >= 0.5, "new", "used")})
Path("../submission").mkdir(exist_ok=True)
archivo = "../submission/submission_p7c_duracion.csv"
submission.to_csv(archivo, index=False)

leida = pd.read_csv(archivo)
assert leida.shape == (30000, 2)
assert list(leida.columns) == ["ID", "condition"]
assert (leida["ID"].to_numpy() == np.arange(1, 30001)).all()
assert leida["condition"].isin(["new", "used"]).all()
print("Formato OK | % new predicho:", round((leida["condition"] == "new").mean(), 4))
anterior = pd.read_csv("../submission/submission_catboost_histgb.csv")
cambian = (anterior["condition"] != leida["condition"]).sum()
print(f"Predicciones distintas vs submission_catboost_histgb.csv (P7c): {cambian} ({cambian / len(leida):.1%})")
extra = (test["duracion_dias"] > 60) & (test["duracion_dias"] <= 60 + 120 / 86400)
print(f"Test con 60 días + segundos: {extra.sum()} filas | predichas new: {(leida['condition'][extra.to_numpy()] == 'new').mean():.1%}")

Formato OK | % new predicho: 0.5289
Predicciones distintas vs submission_catboost_histgb.csv (P7c): 322 (1.1%)
Test con 60 días + segundos: 590 filas | predichas new: 4.1%
